# Model v19: AutoGluon (LightGBM, CatBoost, ExtraTrees) + rete temporale

v19 è la variante "diversa" di v18: stesse feature L, ma nessun XGBoost.

- **Alberi**: AutoGluon con LightGBM, CatBoost ed ExtraTrees, iperparametri fissati a mano e nessun limite di tempo
- **CV**: un predictor per centrale, 5 fold contigui di casi; le predizioni OOF alimentano lo stacking topologico. A differenza di v18 non c'è il gap di 7 giorni tra i fold
- **Rete temporale**: come v18, con più canali e più epoche
- **Blend**: media dei logit, peso scelto sul test locale (score del blend leggermente ottimistico)

Installazione: `pip install "autogluon.tabular[lightgbm,catboost]==1.6.3"`. `lp_cache.npz` è lo stesso di v18; per misurare il tempo reale va cancellato.

### getting the data (+ building the functions and class)

In [ ]:
%pip install "autogluon.tabular[lightgbm, catboost]==1.6.3"

In [ ]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
from tempfile import TemporaryDirectory
from autogluon.tabular import TabularPredictor
from autogluon.features.generators import IdentityFeatureGenerator
import torch
import torch.nn as nn
import scipy.sparse as sp
from scipy.optimize import linprog

In [ ]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)
    xh = win[pos].reshape(len(pos), -1)

    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    all_cols = hourly_cols + vol + [f"{c}_init" for c in wv] + [f"{c}_final" for c in wv]

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(Y.filter(like='result_committed').to_numpy(np.uint8), index=Y.index, columns=Y.columns[2:])
    x = pd.DataFrame(np.hstack([xh, xs]).astype(np.float32), index=Y.index, columns=all_cols)
    return x, y


class DataLoader:
    def __init__(self, data_source):
        self.data_source = Path(data_source)

    def _csv(self, folder, filename, date_columns=()):
        frame = pd.read_csv(self.data_source / folder / filename)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        with (self.data_source / folder / filename).open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv("kernel", "Historical_day_ahead_price_2015_2025.csv", ("date",))

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv("kernel", "Synthetic_water_value_2015_2024.csv", ("date",))

    @cached_property
    def commitment_decision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self):
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")


# Class to create the entire dataset in one go
class Training_set(DataLoader):
    def __init__(self, data_source):
        super().__init__(data_source)
        self.y = None
        self.x = None

    def load_copies(self):
        return {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }

    def create_x(self):
        data = self.load_copies()

        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data: repeat their value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')

        self.y = data.pop('uc').copy()
        x = data.pop('day_ahead_price')
        for name in data.keys():
            x = x.join(data[name].add_suffix('_' + name), how='left')
        self.x = x.copy()

In [ ]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
# vincoli: 6 min_flow + 3 min_volume, serie orarie 2015-2025 (come v6)
C = pd.concat([ts.min_flow_constraint_data.set_index('date'), ts.min_volume_constraint_data.set_index('date')], axis=1).astype(np.float32)
C.index = pd.to_datetime(C.index, utc=True)
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

## Topology and target

In [ ]:
T = 168

# physical constraints from the topological yaml file
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
GEN = {k.lower().replace(' ', '_'): g for k, g in topo['generator'].items()}
PN = {k: g['p_nom'] for k, g in GEN.items()}

# every power plant to its upstream source reservoirs and its downstream receiving reservoir
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['langeidvatn', 'vatjern'], 'vinjevatn'),  # Langeidvatn primo (volume Vatjern costante nei dati)
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}

# linking the downstream plants to their upstream counterparts
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
SRC = {'vinje': 'totak'}
FEED = {'vesle_kjela': ['staavatn'], 'kjela': ['langesae'], 'vinje': ['venemo', 'hyljelihyl']}  # v14: bacini che alimentano i serbatoi a monte

# upstream plants first: their predicted probabilities become features of the downstream plants
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]
auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Feature v4

In [ ]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

lw = lambda a: np.log(np.clip(a, 1, None))  # log senza scala (prezzi/wv < 1 -> 0)

def static(X, plant, logwv=False):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    cols = [f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]
    A0 = X[cols].to_numpy()
    if logwv: A0 = np.where(np.array([not c.endswith('_volume') for c in cols]), lw(A0), A0)
    S = [A0]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wu = X[f'{up[0]}_wv_final'].to_numpy()
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              lw(wu) - lw(wd) if logwv else wu - wd]))
    return np.hstack(S)

def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

## optimizer surrogate

In [ ]:
ETA = 0.88
CAP_RES = {'haukeli': 'vatjern'}

def cap(X, plant):  # portata massima di centrale (Mm3/h)
    up, dn = TOPO[plant]
    r0 = CAP_RES.get(plant, up[0])  # Haukeli prende l'acqua da Vatjern
    hu = head(r0, X[f'{r0}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

CC, _cw = list(C.columns), {}
def cwin(X):
    k = (X.index[0], len(X))
    if k not in _cw:
        pos = C.index.get_indexer(X.index); assert (pos >= 0).all()
        _cw[k] = swv(C.to_numpy(), T, axis=0)[pos]
    return _cw[k]
mvol = lambda X, rs: sum((cwin(X)[:, CC.index(r.capitalize())].max(1) for r in rs if r.capitalize() in CC), np.zeros(len(X), np.float32))
CONS = {'lio': ['r_Vest_Vassdraget', 'b_Byrtevatn', 'b_Vinjevatn', 'Byrtevatn'], 'byrte': ['b_Byrtevatn', 'Byrtevatn'], 'songa': ['Totak'],
        'vinje': ['Totak', 'b_Vinjevatn'], 'vesle_kjela': ['b_Kjelavatn'], 'kjela': ['b_Hyljelihyl'], 'tokke': ['b_Vinjevatn'], 'hogga': ['b_Bandak']}
wsrc = lambda plant, cons: {**SRC, **({'haukeli': 'langeidvatn'} if cons else {})}.get(plant, TOPO[plant][0][0])

def surr(X, plant, rel=None, cons=False):
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant])
    src = wsrc(plant, cons)
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s * 0.0036 = Mm3/h
    Wmax = np.clip(V0 + Win - (mvol(X, up) if cons else 0), 0, T * c)  # massimo utilizzabile
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)  # minimo per non sfiorare
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]
    if rel is not None:
        S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]
    return S

def cons_feats(X, plant, c):
    up, dn = TOPO[plant]; n, W = len(X), cwin(X)
    bc = lambda g: np.repeat(np.asarray(g, np.float32)[:, None], T, 1)
    Vd = X[f'{dn}_volume'].to_numpy() if dn else np.zeros(n)
    Id = inflow(X, [dn]).sum(1) * 0.0036 if dn else np.zeros(n)
    F = [W[:, CC.index(k)] for k in CONS.get(plant, [])]
    F += [bc(sum(X[f'{r}_volume'].to_numpy() for r in up) - mvol(X, up)),
          bc(np.maximum(mvol(X, [dn] if dn else []) - Vd - Id, 0) / c)]
    if plant == 'lio':
        assert 'r_vest_vassdraget_inflow_t0' in X.columns
        w = lambda k: W[:, CC.index(k)]
        need = np.maximum(w('r_Vest_Vassdraget') - inflow(X, ['r_vest_vassdraget']) - w('b_Vinjevatn') - w('b_Byrtevatn'), 0)
        q = (c / 0.0036)[:, None]  # portata massima (m3/s)
        F += [need, np.cumsum(need, 1) / q, bc(need.sum(1) / q[:, 0])]
    return F

## v14: fisica e costi di avvio

Per ogni unità, nell'ordine di accensione: `viterbi` restituisce (miglior piano con unità ON all'ora t) − (miglior piano con unità OFF), al netto di start + stop. Positivo = conviene essere accesa.

In [ ]:
NF0 = 7  # feature per unità in P
NF = {'P': 7, 'L': 9}

def viterbi(m, s):  # m: margine orario di stare ON (casi, T); s: costo di un ciclo start + stop
    h = s / 2  # metà costo per ogni cambio di stato (stessa somma su un ciclo)
    Fon, Fof, Bon, Bof = m.copy(), np.zeros_like(m), np.zeros_like(m), np.zeros_like(m)
    for t in range(1, T):
        Fon[:, t] = m[:, t] + np.maximum(Fon[:, t-1], Fof[:, t-1] - h)
        Fof[:, t] = np.maximum(Fof[:, t-1], Fon[:, t-1] - h)
    for t in range(T - 2, -1, -1):
        Bon[:, t] = np.maximum(m[:, t+1] + Bon[:, t+1], Bof[:, t+1] - h)
        Bof[:, t] = np.maximum(Bof[:, t+1], m[:, t+1] + Bon[:, t+1] - h)
    return Fon + Bon - Fof - Bof

def peak(g):  # portata, salti di riferimento e rendimenti al picco delle curve della turbina
    C_ = g['turb_eff_curves']; b = [int(np.argmax(c['y'])) for c in C_]
    return C_[0]['x'][b[0]], np.array([c['ref'] for c in C_], float), np.array([c['y'][i] / 100 for c, i in zip(C_, b)])

def unit_feats(X, plant, hmin, hmax):
    up, dn = TOPO[plant]; pl = PL[plant]; r0 = CAP_RES.get(plant, up[0])
    P = X[PRICE].to_numpy(); w = X[f'{wsrc(plant, True)}_wv_final'].to_numpy()
    Hg = head(r0, X[f'{r0}_volume'].to_numpy()) - np.maximum(head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0, pl['outlet_line'])
    # soglia lam: water value, spostato solo se il budget d'acqua lo impone
    Ps = -np.sort(-P, 1); pk = lambda h: np.take_along_axis(Ps, np.clip(h.astype(int), 0, T - 1)[:, None], 1)[:, 0]
    lam = np.clip(w, np.where(hmax < T, pk(hmax), -np.inf), np.where(hmin > 0, pk(hmin), np.inf))
    w, lam = w[:, None], lam[:, None]
    wn, ln = np.maximum(w, 0) + 5, np.maximum(lam, 0) + 5
    F, prev = [], 0
    for j, u in enumerate(plant_units[plant]):
        g = GEN[units[u]]; q, refs, et = peak(g)
        Hj = np.maximum(Hg - float(pl['main_loss'][0]) * ((j + 1) * q)**2 - pl['penstock_loss'][g['penstock'] - 1] * q**2, 1)
        tot = 9.81e-3 * (j + 1) * q * Hj * np.interp(Hj, refs, et) * .98
        pm = np.minimum(tot - prev, g['p_max']); prev = tot  # potenza marginale della (j+1)-esima unità
        pref = 9.81e-3 * q * refs.max() * et.max() * .98
        rel, s = (pm / pref)[:, None], (g['startcost_const'] + g['stopcost_const']) / pref  # EUR per MW di riferimento
        D1, D2, D3 = viterbi(P - w, s), viterbi(rel * P - w, s), viterbi(rel * P - lam, s)
        F += [D1 / wn, D2 / wn, D3 / ln, D1 > 0, D3 > 0, np.repeat(rel, T, 1), np.repeat(lw(lam) - lw(w), T, 1)]
    return F

## v17: LP della cascata

In [ ]:
def make_lp():
    K, y = 0.0036, ts.topology_data  # 1 m3/s per 1 h = 0.0036 Mm3
    m, CON = y['model'], y['connections']
    idx_ = lambda d: d.set_index(pd.to_datetime(d['date'], utc=True)).drop(columns='date')
    PR, Wv, Vo, IN, MF, MV = [idx_(d) for d in (ts.day_ahead_price_data, ts.synthetic_water_value_data, ts.volume_data, ts.inflow_data, ts.min_flow_constraint_data, ts.min_volume_constraint_data)]
    PR = PR.iloc[:, 0]
    RS, PS, RV, TN = list(m['reservoir']), list(m['plant']), list(m['river']), list(m['tunnel'])
    gens = {p: [g for g in m['generator'] if g.rsplit('_G', 1)[0] == p] for p in PS}
    FREE = {'Totak_Vaamarvatn', 'Bordalsvatn_Foersvatn', 'Venemo_Totak'}  # gallerie tra bacini: portata nei due versi
    INTAKE = {'Hogga': 'Bandak', 'Tokke': 'Vinjevatn', 'Lio': 'Byrtevatn', 'Byrte': 'Botnedalsvatn', 'Vinje': 'Vaamarvatn', 'Haukeli': 'Vatjern', 'Songa': 'Songavatn', 'Kjela': 'Foersvatn', 'Vesle Kjela': 'Kjelavatn'}
    OUT = {'Hogga': None, 'Tokke': 'Bandak', 'Lio': 'Bandak', 'Byrte': 'Byrtevatn', 'Vinje': 'Vinjevatn', 'Haukeli': 'Vinjevatn', 'Songa': 'Totak', 'Kjela': 'Hyljelihyl', 'Vesle Kjela': 'Foersvatn'}
    hd = lambda r, v: np.interp(v, m['reservoir'][r]['vol_head']['x'], m['reservoir'][r]['vol_head']['y'])

    def seg(p, v0):  # (MW per m3/s marginali, portata massima) di ogni unità
        pl, dn, out, prev = m['plant'][p], OUT[p], [], 0
        Hg = hd(INTAKE[p], v0[INTAKE[p]]) - max(hd(dn, v0[dn]) if dn else 0, pl['outlet_line'])
        for j, g in enumerate(gens[p]):
            G = m['generator'][g]; q, refs, et = peak(G)
            Hj = max(Hg - float(pl['main_loss'][0]) * ((j + 1) * q)**2 - float(pl['penstock_loss'][G['penstock'] - 1]) * q**2, 1)
            tot = 9.81e-3 * (j + 1) * q * Hj * np.interp(Hj, refs, et) * .98
            out.append(((tot - prev) / q, max(G['turb_eff_curves'][0]['x']))); prev = tot
        return out

    # variabili per ora: segmenti | fiumi e gallerie | volumi | scarti dei vincoli minimi
    segs = [(p, j) for p in PS for j in range(len(gens[p]))]; si = {s: i for i, s in enumerate(segs)}
    Fl = RV + TN; nS, nF = len(segs), len(Fl); fi = {o: nS + i for i, o in enumerate(Fl)}; ri = {r: nS + nF + i for i, r in enumerate(RS)}
    slk = list(MF.columns) + list(MV.columns); sli = {o: nS + nF + len(RS) + i for i, o in enumerate(slk)}; NV = nS + nF + len(RS) + len(slk)
    at = lambda v, t: v + t * NV
    expr = lambda o: [si[(o, j)] for j in range(len(gens[o]))] if o in m['plant'] else [fi[o]]
    inp, out = {}, {}
    for c in CON:
        a, b = c['from'], c['to']
        if c['from_type'] == 'generator': continue
        if c['to_type'] != 'reservoir' and c['from_type'] == 'reservoir': out.setdefault(a, []).append(b)
        else: inp.setdefault(b, []).append(a)
    # bilanci
    R, Cc, Va, n, bal, nodes = [], [], [], 0, [], []
    def add(r, v, x): R.append(r); Cc.append(v); Va.append(x)
    for r in RS:
        for t in range(T):
            add(n, at(ri[r], t), 1.)
            if t: add(n, at(ri[r], t - 1), -1.)
            for o, sg in [(o, -K) for o in inp.get(r, [])] + [(o, K) for o in out.get(r, [])]:
                for v in expr(o): add(n, at(v, t), sg)
            bal.append((n, r, t)); n += 1
    for o in [o for o in inp if o not in m['reservoir']]:
        for t in range(T):
            for v in expr(o): add(n, at(v, t), 1.)
            for u in inp[o]:
                for v in expr(u): add(n, at(v, t), -1.)
            nodes.append((n, o, t)); n += 1
    Aeq = sp.csr_matrix((Va, (R, Cc)), shape=(n, NV * T))
    # vincoli minimi con scarto
    R, Cc, Va, n, ubk = [], [], [], 0, []
    for o in slk:
        for t in range(T):
            add(n, at(fi[o] if o in fi else ri[o], t), -1.); add(n, at(sli[o], t), -1.); ubk.append((o, t)); n += 1
    Aub = sp.csr_matrix((Va, (R, Cc)), shape=(n, NV * T))

    def solve(ts_):
        p0 = PR.index.get_loc(ts_); P = PR.values[p0:p0 + T]; v0 = Vo.loc[ts_]
        wf = Wv.reindex([ts_ + pd.Timedelta(days=7)], method='ffill').iloc[0]
        loc = lambda d: d.iloc[d.index.get_loc(ts_):][:T]
        I, mf, mv = loc(IN), loc(MF), loc(MV)
        c, lo, hi, es = np.zeros(NV * T), np.zeros(NV * T), np.full(NV * T, np.inf), {}
        for p in PS:
            S = seg(p, v0)
            for j, (e, cp) in enumerate(S):
                es[(p, j)] = e; v = at(si[(p, j)], np.arange(T))
                c[v], hi[v] = -(P * e - wf[INTAKE[p]] * S[0][0]), cp  # l'acqua rilasciata costa il water value del proprio bacino
        for o in RV:
            v = at(fi[o], np.arange(T)); ro = m['river'][o]
            c[v], hi[v] = ro.get('flow_cost_const', 2. if 'flow_cost_curve' in ro else 0.), ro.get('max_flow_const', np.inf)
        for o in TN:
            v = at(fi[o], np.arange(T)); hi[v] = 500
            if o in FREE: lo[v] = -500
        for r in RS: hi[at(ri[r], np.arange(T))] = m['reservoir'][r]['vol_head']['x'][-1]
        for o in slk: c[at(sli[o], np.arange(T))] = 1e4
        b = np.zeros(Aeq.shape[0])
        for row, r, t in bal: b[row] = K * I[r].values[t] + (float(v0[r]) if t == 0 else 0)
        for row, o, t in nodes:
            if o in I: b[row] = I[o].values[t]
        bu = np.array([-(mf[o].values[t] if o in mf else mv[o].values[t]) for o, t in ubk])
        r = linprog(c, A_ub=Aub, b_ub=bu, A_eq=Aeq, b_eq=b, bounds=np.c_[lo, hi], method='highs'); assert r.status == 0, r.message
        rc = r.lower.marginals + r.upper.marginals
        out_ = np.zeros((len(units), 2, T), np.float32)
        for p, j in segs:
            v = at(si[(p, j)], np.arange(T)); u = units.index(gens[p][j].lower().replace(' ', '_'))
            out_[u] = r.x[v] / hi[v], -rc[v] / es[(p, j)]  # portata relativa, costo ridotto in EUR/MWh
        return out_
    return solve

_lp, _lps, LPK = None, {}, 1.  # LPK: scala del costo ridotto quando i prezzi sono riscalati
def lp_feats(X):  # (casi, unità, 2, T), con cache su disco
    global _lp
    f = Path(DATA) / 'lp_cache.npz'
    if not _lps and f.exists(): _lps.update(zip(np.load(f)['k'].tolist(), np.load(f)['v']))
    miss = [t for t in X.index if str(t) not in _lps]
    if miss:
        _lp = _lp or make_lp()
        for t in miss: _lps[str(t)] = _lp(t)
        k = list(_lps); np.savez(f, k=np.array(k), v=np.stack([_lps[a] for a in k]))
    return np.stack([_lps[str(t)] for t in X.index])

## feature di centrale

In [ ]:
def base(X, plant, up_prob=None, rel=None, var='N'):
    up, dn, n = TOPO[plant][0], TOPO[plant][1], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    rank, drank = P.argsort(1).argsort(1) / (T - 1), D.argsort(2).argsort(2).reshape(n, T) / 23
    dmean, wmean = np.repeat(D.mean(2), 24, 1), P.mean(1, keepdims=True)
    rl, sd = lw(P) - lw(wmean), P.std(1, keepdims=True) + 1  # senza scala: livelli relativi, z-score, log(P/wv)
    PF = [rl, shift(rl, 1), shift(rl, -1), rank, drank, (P - dmean) / sd, (P - wmean) / sd, lw(P) - lw(wvf)]
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7] + PF + [I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    dif = lambda r, w: lw(P) - lw(X[f'{r}_wv_{w}'].to_numpy()[:, None])
    F += cons_feats(X, plant, cap(X, plant)) + [dif(r, w) for r in up + ([dn] if dn else []) for w in ('init', 'final')]
    c = cap(X, plant)[:, None]
    ci = np.cumsum(I, 1) * 0.0036
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)[:, None]
    F += [ci / c, (V0 + ci - sum(RES[r]['max_vol'] for r in up)) / c]  # inflow cumulato e pressione di sfioro
    S = surr(X, plant, rel, True)
    F += S
    A = [np.repeat(static(X, plant, True), T, 0), np.repeat(X.index.dayofyear.to_numpy(), T)]
    if var in ('P', 'L'):
        fr = FEED.get(plant, [])
        feed = sum((X[f'{r}_volume'].to_numpy() + inflow(X, [r]).sum(1) * 0.0036 for r in fr), np.zeros(n)) / c[:, 0]
        U = unit_feats(X, plant, S[4][:, 0] * T, S[7][:, 0] * T + feed)  # S[4], S[7]: Wmin e Wmax in frazione di settimana
        A.append(np.repeat(feed, T))
        if var == 'L':  # stesse NF per unità più portata e costo ridotto dell'LP, fatte con il water value per non dipendere dalla scala
            G, wn = lp_feats(X), np.maximum(X[f'{wsrc(plant, True)}_wv_final'].to_numpy(), 0)[:, None] + 5
            U = [v for j, u in enumerate(plant_units[plant]) for v in U[j * NF0:(j + 1) * NF0] + [G[:, u, 0], G[:, u, 1] * LPK / wn]]
    else: U = []
    # colonne: comuni | NF per unità | S[2] (ultima, usata da design)
    return np.column_stack([f.ravel() for f in F] + A + [f.ravel() for f in U] + [S[2].ravel()]).astype(np.float32)

# training

Come v12: un modello per centrale con unità impilate e stacking topologico. In `design` ogni unità riceve solo le proprie `NF` feature. La media dei modelli dei fold predice i casi nuovi, quindi non serve un refit finale.

In [ ]:
FOLDS, CPUS = 5, 4
HP = {'GBM': {'num_boost_round': 300, 'learning_rate': 0.05, 'num_leaves': 63, 'feature_fraction': 0.5, 'bagging_fraction': 0.8, 'bagging_freq': 1},
      'CAT': {'iterations': 300, 'learning_rate': 0.05, 'depth': 6},
      'XT': {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 20, 'max_features': 0.3}}

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, var):
    r = ridx(c); R = B[r]
    k = NF.get(var, 0); s0 = R.shape[1] - 1 - k * len(us)  # inizio delle feature per unità
    def blk(j, u):
        return np.column_stack([R[:, :s0], R[:, s0 + j * k: s0 + (j + 1) * k], np.full(len(r), u), np.full(len(r), j), R[:, -1] - j])
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, var, cw=None):
    n, nt = len(B) // T, len(Bt) // T
    y = ylong(np.arange(n), us)
    if y.min() == y.max():
        v = float(np.clip(y[0], .01, .99))
        return np.full((len(us), n, T), v), np.full((len(us), nt, T), v)
    Xa, Xb = design(B, np.arange(n), us, var), design(Bt, np.arange(nt), us, var)
    cols = [f'f{i}' for i in range(Xa.shape[1])]
    rows = lambda a: np.tile(np.repeat(a, T), len(us))  # valore per caso -> righe
    d = pd.DataFrame(Xa, columns=cols).assign(y=y, fold=rows(np.arange(n) * FOLDS // n))
    if cw is not None: d['w'] = rows(cw)
    with TemporaryDirectory() as p:
        m = TabularPredictor(label='y', problem_type='binary', eval_metric='roc_auc', groups='fold', sample_weight='w' if cw is not None else None, path=f'{p}/m', verbosity=1).fit(
            d, hyperparameters=HP, num_stack_levels=0, dynamic_stacking=False, num_cpus=CPUS, calibrate=False,
            feature_generator=IdentityFeatureGenerator(), ag_args_ensemble={'fold_fitting_strategy': 'sequential_local'})
        print(m.leaderboard(silent=True)[['model', 'score_val', 'fit_time']].round(4).to_string(index=False))
        oo = m.predict_proba_oof(as_multiclass=False).to_numpy().reshape(len(us), n, T)
        te = m.predict_proba(pd.DataFrame(Xb, columns=cols), as_multiclass=False).to_numpy().reshape(len(us), nt, T)
    return oo, te

def run(VAR, xtr, xte, rw=1.0):
    cw = None if rw == 1 else np.where(xtr.index.year >= 2021, rw, 1.0).astype(np.float32)
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32); O = np.zeros((len(xtr), len(units), T), np.float32)
    for plant in ORDER:
        us, ups, v = plant_units[plant], UP_PLANTS.get(plant, []), VAR[plant]
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if ups else None
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr), v), base(xte, plant, m(te), s(rte), v), us, v, cw)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok')
    return P, O

## AutoGluon (variante L)

In [ ]:
def cached(name, fn):
    f = Path(DATA) / f'preds_v19_{name}.npz'
    if f.exists(): return np.load(f)['P']
    p = fn(); np.savez_compressed(f, P=p)
    return p

VAR = {p: 'L' for p in ORDER}
Pag = cached('ag_test', lambda: run(VAR, x_train, x_test)[0])
print(f'AutoGluon test {auc(Y3te, Pag):.4f}')

## rete temporale

In [ ]:
EPOCHS, BATCH, CH, LR, DROP, TSEEDS = 15, 32, 192, 1e-3, 0.2, 3
SCALES = (1, .5, 2, 4)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'

def tensor(X, k=1.):  # (casi, canali, T): tutte le feature L di tutte le centrali, senza stacking
    global LPK
    LPK = k
    A = np.hstack([base(X, p, None, None, 'L') for p in ORDER])
    LPK = 1.
    return A.reshape(len(X), T, -1).transpose(0, 2, 1)

def scaled(X, k):  # stessi casi con prezzi e water value moltiplicati per k
    Z = X.copy(); cols = PRICE + [c for c in X.columns if c.endswith(('_wv_init', '_wv_final'))]
    Z[cols] = Z[cols] * k
    return Z

class Block(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.net = nn.Sequential(nn.GroupNorm(8, c), nn.Conv1d(c, c, 3, padding=d, dilation=d), nn.GELU(), nn.Dropout(DROP), nn.Conv1d(c, c, 1))
    def forward(self, x):
        return x + self.net(x)

class TCN(nn.Module):  # convoluzioni non causali: la settimana è nota in anticipo
    def __init__(self, nf, nu):
        super().__init__()
        self.net = nn.Sequential(nn.Dropout1d(DROP), nn.Conv1d(nf, CH, 1), *[Block(CH, d) for d in (1, 2, 4, 8, 24, 48, 72)], nn.Conv1d(CH, nu, 1))
    def forward(self, x):
        return self.net(x)

def fit_tcn(xtr, Ytr, xp, cw=None, Yp=None):  # xtr, xp: DataFrame dei casi
    A = tensor(xtr); sd = A.std((0, 2), dtype=np.float64); keep = sd > 1e-4  # float64: niente rumore di arrotondamento sulle colonne costanti
    mu, sd = A.mean((0, 2), dtype=np.float64)[keep][None, :, None].astype(np.float32), sd[keep][None, :, None].astype(np.float32)
    tt = lambda A: torch.tensor(np.clip((A[:, keep] - mu) / sd, -8, 8), dtype=torch.float32)
    n, S = len(xtr), len(SCALES); del A
    Xtr = torch.zeros((n * S, int(keep.sum()), T), dtype=torch.float16)  # float16: le copie riscalate occupano metà memoria
    for i, k in enumerate(SCALES): Xtr[i * n:(i + 1) * n] = tt(tensor(scaled(xtr, k), k)).half()
    Xp, Y = tt(tensor(xp)), torch.tensor(Ytr, dtype=torch.float32).repeat(S, 1, 1)
    w = (torch.ones(n) if cw is None else torch.tensor(cw, dtype=torch.float32)).repeat(S)
    def predict(m):
        m.eval()
        with torch.no_grad():
            return np.array(torch.cat([m(Xp[i:i + 128].to(DEV)).cpu() for i in range(0, len(Xp), 128)]).tolist(), dtype=np.float32)  # tolist: evita il ponte torch-numpy se le versioni non coincidono
    logits = 0
    for seed in range(TSEEDS):
        torch.manual_seed(seed)
        m = TCN(Xtr.shape[1], Y.shape[1]).to(DEV)
        opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-2)
        steps = EPOCHS * int(np.ceil(len(Xtr) / BATCH))
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, LR, total_steps=steps)
        for ep in range(EPOCHS):
            m.train(); tot = 0
            for b in torch.randperm(len(Xtr)).split(BATCH):
                l = (nn.functional.binary_cross_entropy_with_logits(m(Xtr[b].float().to(DEV)), Y[b].to(DEV), reduction='none').mean((1, 2)) * w[b].to(DEV)).mean()
                opt.zero_grad(); l.backward(); opt.step(); sch.step(); tot += l.item() * len(b)
            if Yp is not None and ((ep + 1) % 2 == 0 or ep == EPOCHS - 1):  # solo diagnostica: gli epoch non si scelgono sul test
                print(f'seed {seed} ep {ep + 1} loss {tot / len(Xtr):.4f} test {auc(Yp, predict(m)):.4f}')
        logits = logits + predict(m) / TSEEDS
    return 1 / (1 + np.exp(-logits))

In [ ]:
Ptcn = cached('tcn_test', lambda: fit_tcn(x_train, Y3, x_test, Yp=Y3te))
print(f'TCN test {auc(Y3te, Ptcn):.4f}')

## blend

In [ ]:
lg = lambda p: np.log(np.clip(p, 1e-5, 1 - 1e-5) / (1 - np.clip(p, 1e-5, 1 - 1e-5)))
blend = lambda a, b, w: (1 - w) * lg(a) + w * lg(b)
res = {w: auc(Y3te, blend(Pag, Ptcn, w)) for w in np.arange(0, 1.01, .1).round(1)}
WB = max(res, key=res.get)
print(res); print('peso TCN migliore:', WB)
pd.DataFrame({'ag': [auc(Y3te[:, i], Pag[:, i]) for i in range(len(units))], 'tcn': [auc(Y3te[:, i], Ptcn[:, i]) for i in range(len(units))],
              'blend': [auc(Y3te[:, i], blend(Pag, Ptcn, WB)[:, i]) for i in range(len(units))]}, index=units).round(4)

## kaggle submission

In [ ]:
MAP, SS = f'{DATA}/prediction_mapping.csv', f'{DATA}/sample_submission.csv'  # adattare ai percorsi reali
mp = pd.read_csv(MAP); mp['starttime'] = pd.to_datetime(mp['starttime'], utc=True)
dummy = pd.DataFrame(0, index=pd.DatetimeIndex(mp['starttime']), columns=['Run No', 'total_calculation_time', *y_train.columns])
x_all, y_all = build_cases(X_all, Y_all)
x_kag, _ = build_cases(X_all, dummy)
assert len(x_all) == 2922 and len(x_kag) == len(mp)
Y3 = stack3(y_all); Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]  # run() usa questi target globali
RW = 2.0  # peso casi 2021+
Pk_x = cached('ag_kaggle', lambda: run(VAR, x_all, x_kag, RW)[0])
cw = np.where(x_all.index.year >= 2021, RW, 1.0)
Pk_t = cached('tcn_kaggle', lambda: fit_tcn(x_all, Y3, x_kag, cw))
Pk = 1 / (1 + np.exp(-blend(Pk_x, Pk_t, WB)))
sub = pd.DataFrame(Pk.reshape(len(Pk), -1), columns=out_cols)[list(y_train.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
ss = pd.read_csv(SS); norm = lambda c: c.lower().replace(' ', '_'); nm = {norm(c): c for c in ss.columns}
sub = sub.rename(columns=lambda c: nm.get(norm(c), c))  # nomi colonna come in sample_submission.csv
assert set(sub.columns) == set(ss.columns), set(sub.columns) ^ set(ss.columns)
sub = sub.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]
sub.to_csv(f'{DATA}/submission_v19.csv', index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))